# Cross-corpus spam detection: A→B, B→A, A→C, B→C

Five classical classifiers × three feature configurations (text / metadata / text+metadata). Each classifier is trained once on the 27,000-email source subset and evaluated once on the fixed target sample.

**Input:** the five deposited `.pkl` files listed in `FILES`. They are the output of the separate email preprocessing pipeline (cleaned `text`, six metadata columns, `label`); no preprocessing is done in this notebook.

**How to run:** put the notebook next to the `.pkl` files (or set `DATA_DIR`), then *Run All*. Set `CHECK_ONLY = True` first to validate the files without training.

**Outputs** (`outputs_cross_corpus/`): `cross_corpus_results.csv`, `cross_corpus_predictions.csv`, `run_metadata.json`.

In [ ]:
import json
import pickle
import platform
import time
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             matthews_corrcoef, precision_score, recall_score)
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

## 0. Configuration

In [ ]:
SEED = 42
OUT_DIR = Path("outputs_cross_corpus")
DATA_DIR = Path(".")                      # folder holding the deposited .pkl files
ONLY = None                               # e.g. ["A->B", "B->C"] to run a subset; None = all four
CHECK_ONLY = False                        # True = load and validate the data, no training

TEXT_COL = "text"
LABEL_COL = "label"                      # 1 = spam, 0 = legitimate (ham)
META_COLS = ["num_exclamations", "num_dollars", "num_urls",
             "num_emails", "num_hashes", "num_digits"]

TFIDF_PARAMS = dict(max_features=20000, ngram_range=(1, 2))   # fitted on source training data only
FEATURE_CONFIGS = ["text", "metadata", "text_metadata"]
MODELS_TO_RUN = ["Naive Bayes", "Linear SVM", "Logistic Regression", "XGBoost", "Random Forest"]
SAVE_PREDICTIONS = True                  # needed for exact confusion matrices and bootstrap CIs


N_TRAIN, N_TARGET = 27000, 9956          # expected sizes (checked before any model is trained)

# Exact deposited filenames
FILES = dict(
    A_train="dataset_a_cross_train_27000_ids.pkl",
    B_train="dataset_b_cross_train_27000_ids.pkl",
    A_target="dataset_a_to_b_target_9956_ids.pkl",   # Dataset A sample = target of B->A (inferred, please confirm)
    B_target="dataset_b_to_a_target_9956_ids.pkl",   # Dataset B sample = target of A->B (inferred, please confirm)
    C_target="datasetC_9956_ids.pkl",
)

EXPERIMENTS = [
    dict(source="A", target="B", train="A_train", test="B_target"),
    dict(source="B", target="A", train="B_train", test="A_target"),
    dict(source="A", target="C", train="A_train", test="C_target"),
    dict(source="B", target="C", train="B_train", test="C_target"),
]

## 1. Loading
Preprocessing is done upstream by the email preprocessing pipeline.

In [ ]:
def load_corpus(path):
    """Load one deposited, already-preprocessed file -> dataframe (text, META_COLS, label, record_id)."""
    path = Path(path)
    df = pd.read_csv(path) if path.suffix == ".csv" else pd.read_pickle(path)
    if not isinstance(df, pd.DataFrame):
        raise TypeError(f"{path.name} holds {type(df).__name__}, expected the preprocessed DataFrame "
                        f"(text, {', '.join(META_COLS)}, label).")
    missing = [c for c in [TEXT_COL, LABEL_COL, *META_COLS] if c not in df.columns]
    if missing:
        raise ValueError(f"{path.name} is missing columns {missing}. Run it through the email "
                         f"preprocessing pipeline first.")
    if "record_id" not in df.columns:                     # fall back to the index as the id
        df = df.assign(record_id=df.index.values)
    df = df.reset_index(drop=True)
    df[TEXT_COL] = df[TEXT_COL].fillna("").astype(str)    # TF-IDF cannot take NaN (empty after cleaning)
    return df

## 2. Feature configurations

In [ ]:
def build_preprocessor(config):
    """ColumnTransformer for one feature configuration. The vectoriser is fitted on the training data only."""
    if config == "text":
        return ColumnTransformer([("text", TfidfVectorizer(**TFIDF_PARAMS), TEXT_COL)])
    if config == "metadata":
        return ColumnTransformer([("meta", "passthrough", META_COLS)])
    if config == "text_metadata":
        return ColumnTransformer([("text", TfidfVectorizer(**TFIDF_PARAMS), TEXT_COL),
                                  ("meta", "passthrough", META_COLS)])
    raise ValueError(config)

## 3. Models
Same hyperparameters as the within-corpus experiments.

In [ ]:
def build_models():
    models = {
        "Naive Bayes": MultinomialNB(alpha=1.0),
        "Linear SVM": LinearSVC(C=1.0, max_iter=5000, random_state=SEED),
        "Logistic Regression": LogisticRegression(solver="saga", C=0.1, max_iter=20000, tol=1e-3,
                                                  random_state=SEED, n_jobs=-1),
        "Random Forest": RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=1),
    }
    if "XGBoost" in MODELS_TO_RUN:
        from xgboost import XGBClassifier
        models["XGBoost"] = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1, subsample=0.8,
                                          colsample_bytree=0.8, random_state=SEED, n_jobs=-1,
                                          eval_metric="logloss")
    return {k: models[k] for k in MODELS_TO_RUN}

## 4. Evaluation of one run

In [ ]:
def evaluate(model, config, X_train, y_train, X_test, y_test):
    pipe = Pipeline([("features", build_preprocessor(config)), ("classifier", model)])

    t0 = time.perf_counter()
    pipe.fit(X_train, y_train)
    train_time = time.perf_counter() - t0

    t0 = time.perf_counter()
    pred = pipe.predict(X_test)
    infer_time = time.perf_counter() - t0

    tn, fp, fn, tp = confusion_matrix(y_test, pred, labels=[0, 1]).ravel()
    metrics = dict(
        accuracy=accuracy_score(y_test, pred),
        precision=precision_score(y_test, pred, zero_division=0),
        recall=recall_score(y_test, pred, zero_division=0),
        f1=f1_score(y_test, pred, zero_division=0),
        mcc=matthews_corrcoef(y_test, pred),
        specificity=tn / (tn + fp), fpr=fp / (tn + fp),
        TN=tn, FP=fp, FN=fn, TP=tp,
        train_time_sec=train_time,
        latency_ms_per_email=infer_time / len(X_test) * 1000,
        model_size_mb=len(pickle.dumps(pipe)) / (1024 * 1024),
    )
    return metrics, pred

## 5. Run the experiments

In [ ]:
def check_files(data_dir, experiments):
    """Fail early (before any training) if a deposited file is missing."""
    needed = sorted({FILES[e[k]] for e in experiments for k in ("train", "test")})
    missing = [f for f in needed if not (data_dir / f).exists()]
    if missing:
        raise FileNotFoundError("Missing files in %s:\n  %s" % (data_dir.resolve(), "\n  ".join(missing)))


def validate(df, name, expected_n):
    assert set(df[LABEL_COL].unique()) <= {0, 1}, f"{name}: labels must be 0/1"
    if len(df) != expected_n:
        print(f"  WARNING: {name} has n={len(df)}, expected {expected_n}")

In [ ]:
exps = [e for e in EXPERIMENTS if not ONLY or f"{e['source']}->{e['target']}" in ONLY]
check_files(DATA_DIR, exps)
OUT_DIR.mkdir(exist_ok=True)
results, predictions = [], []

for exp in exps:
    src, tgt = exp["source"], exp["target"]
    print(f"\n=== {src} -> {tgt} ===")
    train = load_corpus(DATA_DIR / FILES[exp["train"]])
    test = load_corpus(DATA_DIR / FILES[exp["test"]])
    validate(train, FILES[exp["train"]], N_TRAIN)
    validate(test, FILES[exp["test"]], N_TARGET)
    X_train, y_train = train[[TEXT_COL] + META_COLS], train[LABEL_COL].astype(int)
    X_test, y_test = test[[TEXT_COL] + META_COLS], test[LABEL_COL].astype(int)
    print(f"  train n={len(train)} (spam share {y_train.mean():.4f}) | "
          f"test n={len(test)} (spam share {y_test.mean():.4f})")
    if CHECK_ONLY:
        continue

    for config in FEATURE_CONFIGS:
        for name, model in build_models().items():
            print(f"  [{config}] {name} ...", end=" ", flush=True)
            m, pred = evaluate(model, config, X_train, y_train, X_test, y_test)
            print(f"F1={m['f1']:.4f}")
            results.append(dict(source=src, target=tgt, features=config, model=name, **m))
            if SAVE_PREDICTIONS:
                predictions.append(pd.DataFrame(dict(source=src, target=tgt, features=config, model=name,
                                                     id=test["record_id"].values, y=y_test.values, p=pred)))
    # save after every experiment so a crash does not lose finished runs
    if results:
        pd.DataFrame(results).to_csv(OUT_DIR / "cross_corpus_results.csv", index=False)
        if SAVE_PREDICTIONS:
            pd.concat(predictions).to_csv(OUT_DIR / "cross_corpus_predictions.csv", index=False)

print("\nAll files found and validated." if CHECK_ONLY else "\nAll experiments finished.")

## 6. Save run metadata and summarise

In [ ]:
meta = dict(seed=SEED, tfidf=TFIDF_PARAMS, meta_cols=META_COLS, feature_configs=FEATURE_CONFIGS,
            files=FILES, experiments=exps, python=platform.python_version(), numpy=np.__version__,
            pandas=pd.__version__, sklearn=sklearn.__version__)
try:
    import xgboost
    meta["xgboost"] = xgboost.__version__
except ImportError:
    pass
(OUT_DIR / "run_metadata.json").write_text(json.dumps(meta, indent=2, default=str))
print("Saved ->", OUT_DIR.resolve())

In [ ]:
res = pd.read_csv(OUT_DIR / "cross_corpus_results.csv")
res.pivot_table(index=["source", "target", "model"], columns="features", values="f1").round(4)

## 7. Optional diagnostic: do the pipeline's features match a manual build?
The same `X` (text + six metadata columns) goes into every configuration; each `ColumnTransformer` selects its own columns. This cell rebuilds the features the manual way (text only / metadata only / TF-IDF stacked with metadata) and reports any difference. It also lists extra columns in the files, which would be included if X were 'everything except the label'.

In [ ]:
from scipy import sparse

def equivalence_check(train, test):
    """Compare the pipeline's feature matrices with the 'manual' way: text Series -> TF-IDF,
    metadata DataFrame as is, combination = TF-IDF hstack metadata."""
    extra = [c for c in train.columns if c not in [TEXT_COL, LABEL_COL, *META_COLS]]
    print("Columns in the file beyond text / label / the six metadata columns:", extra or "none")
    print("   (if you used 'everything except the label' as X, these were part of your features)\n")
    cols = [TEXT_COL] + META_COLS
    for config in FEATURE_CONFIGS:
        Xa = build_preprocessor(config).fit(train[cols]).transform(test[cols])
        if config == "metadata":
            Xb = test[META_COLS].values
        else:
            Xt = TfidfVectorizer(**TFIDF_PARAMS).fit(train[TEXT_COL]).transform(test[TEXT_COL])
            Xb = Xt if config == "text" else sparse.hstack([Xt, sparse.csr_matrix(test[META_COLS].values)])
        diff = abs(sparse.csr_matrix(Xa) - sparse.csr_matrix(Xb)).max()
        print(f"{config:14s} pipeline {Xa.shape} | manual {Xb.shape} | max abs difference {diff:g}")

In [ ]:
train = load_corpus(DATA_DIR / FILES["B_train"])
test = load_corpus(DATA_DIR / FILES["C_target"])
equivalence_check(train, test)

## 8. Optional diagnostic: are the deposited files the ones used for the initial results?
Compares an earlier file (e.g. `A_train_27000.pkl`) with its deposited counterpart, and shows which corpus each target file contains.

In [ ]:
OLD_TO_NEW = {"A_train_27000.pkl": "A_train"}      # add more pairs if you have the old files
for old_name, key in OLD_TO_NEW.items():
    if not (DATA_DIR / old_name).exists():
        print(f"{old_name}: not found, skipped"); continue
    old = pd.read_pickle(DATA_DIR / old_name).reset_index(drop=True)
    new = load_corpus(DATA_DIR / FILES[key])
    print(f"{old_name} vs {FILES[key]}")
    print("  shape", old.shape, new.shape, "| column differences:", set(old.columns) ^ set(new.columns))
    print("  same texts (any order):", set(old[TEXT_COL]) == set(new[TEXT_COL]),
          "| identical row order:", old[TEXT_COL].equals(new[TEXT_COL]))
    print("  same labels per text:", old.groupby(TEXT_COL)[LABEL_COL].first().equals(new.groupby(TEXT_COL)[LABEL_COL].first()))